# EASE (baseline bổ sung) trên Kaggle CPU

EASE chính xác trên đủ catalog 44.8k item, 4 mật độ × seed 42, một cấu hình cố định (`configs/ease.yaml`).
Cần khoảng **25 GB RAM trống**, nên chạy trên Kaggle thay vì máy local. Không dùng GPU.

**Cách chạy**
1. Settings: Accelerator **None (CPU)**, Internet **On**.
2. **Save Version → Save & Run All (Commit)**.
3. Tải **`ease_ket_qua.zip`** ở cell cuối (hoặc tab Output), giải nén vào thư mục repo trên máy:
   nó chỉ chứa `results/ease/...`, không ghi đè kết quả của 5 mô hình kia.
   Sau đó chạy `python scripts\summarize_results.py` trên máy để cập nhật `results/summary.csv`.

Nếu RAM trống dưới mức cần, `train_model.py` dừng với `MemoryError` (không thu nhỏ catalog).


In [ ]:
import os, subprocess, sys, time, zipfile
from pathlib import Path

import psutil

REPO_URL = "https://github.com/tuyenda2011/SGKR-SPHR-RecSys.git"
CODE = Path("/tmp/SGKR-SPHR-RecSys")          # code + data (not kept in the output)
RESULTS = Path("/kaggle/working/results")     # kept as the notebook output
DENSITIES = ["0.25", "0.5", "0.75", "1.0"]
print(f"RAM: total {psutil.virtual_memory().total / 2**30:.1f} GiB, "
      f"available {psutil.virtual_memory().available / 2**30:.1f} GiB, CPUs {os.cpu_count()}")

In [ ]:
if not CODE.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(CODE)], check=True)
commit = subprocess.run(["git", "rev-parse", "HEAD"], cwd=CODE, capture_output=True, text=True, check=True).stdout.strip()
Path("/kaggle/working/code_commit.txt").write_text(commit + "\n")
print("code commit:", commit)
RESULTS.mkdir(parents=True, exist_ok=True)

In [ ]:
start = time.time()
log_path = Path("/kaggle/working/ease.log")
with log_path.open("a", encoding="utf-8") as log:
    for density in DENSITIES:
        t0 = time.time()
        code = subprocess.run([sys.executable, "-u", "scripts/train_model.py", "--model", "ease", "--density", density,
                               "--results", str(RESULTS)], cwd=CODE, stdout=log, stderr=subprocess.STDOUT).returncode
        print(f"d{round(float(density) * 100)}: exit {code}, {(time.time() - t0) / 60:.1f} min", flush=True)
print(f"total {(time.time() - start) / 60:.1f} min")
print("".join(log_path.read_text(encoding="utf-8", errors="replace").splitlines(keepends=True)[-6:]))

In [ ]:
import json
from IPython.display import FileLink, display

for path in sorted(RESULTS.glob("ease/d*/seed*/metrics.json")):
    run = json.loads(path.read_text(encoding="utf-8"))
    print(f"ease d{round(run['density'] * 100)}: NDCG@20={run['val_metrics']['NDCG@20']:.5f} "
          f"Recall@20={run['val_metrics']['Recall@20']:.5f} fit={run['train_seconds'] / 60:.1f} min")

# Only results/ease/** (metrics, predictions, logs): extracting it locally cannot overwrite the other models.
archive = Path("/kaggle/working/ease_ket_qua.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zf:
    for item in RESULTS.glob("ease/**/*"):
        if item.is_file():
            zf.write(item, Path("results") / item.relative_to(RESULTS))
    zf.write("/kaggle/working/ease.log", "logs/ease.log")
print(f"Archive: {archive} ({archive.stat().st_size / 2**20:.1f} MiB)")
display(FileLink(archive.name))